# LLM Fine-Tuning Demo

This notebook demonstrates the key features of the LLM fine-tuning framework.

## Contents
1. Basic Setup
2. Model Loading
3. LoRA Application
4. Dataset Preparation
5. Training
6. Time Series Forecasting

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import torch
import numpy as np
import matplotlib.pyplot as plt
import logging

logging.basicConfig(level=logging.INFO)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load a Small LLM

In [ ]:
from src.core.model_loader import ModelLoader

# Initialize loader
loader = ModelLoader()

# Load GPT-2
model, tokenizer = loader.load_model(
    model_name="gpt2",
    quantization=None,  # Set to "4bit" if you have CUDA
)

print("\nModel loaded successfully!")
loader.print_trainable_parameters(model)

## 3. Apply LoRA for Efficient Fine-Tuning

In [ ]:
from src.core.lora import LoRAConfig, apply_lora

# Create LoRA configuration
lora_config = LoRAConfig(
    r=8,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj"],  # GPT-2 attention modules
    lora_dropout=0.1,
)

# Apply LoRA
model = apply_lora(model, lora_config, prepare_for_quantization=False)

print("\nLoRA applied! Trainable parameters now:")
loader.print_trainable_parameters(model)

## 4. Prepare Dataset

In [ ]:
from src.datasets.base_loader import prepare_dataset

# Load a small subset for demo
train_dataset = prepare_dataset(
    dataset_name="wikitext",
    tokenizer=tokenizer,
    split="train[:500]",  # Small subset
    text_column="text",
    max_length=256,
    subset="wikitext-2-raw-v1",
)

print(f"\nDataset prepared: {len(train_dataset)} examples")
print(f"First example keys: {train_dataset[0].keys()}")

## 5. Train the Model

Note: This is a small demo. Real training would use more data and epochs.

In [ ]:
from src.core.trainer import FineTuner

# Create trainer
trainer = FineTuner(
    model=model,
    tokenizer=tokenizer,
    output_dir="./notebook_outputs",
)

# Train for 1 epoch as demo
result = trainer.train(
    train_dataset=train_dataset,
    num_epochs=1,
    batch_size=2,
    learning_rate=2e-4,
    gradient_accumulation_steps=4,
    logging_steps=10,
)

print("\nTraining completed!")

## 6. Time Series Forecasting

Now let's demonstrate financial time series prediction.

In [ ]:
from src.timeseries.financial_preprocessor import FinancialDataPreprocessor

# Initialize preprocessor
preprocessor = FinancialDataPreprocessor(
    sequence_length=30,
    prediction_horizon=1,
    return_type="log",
)

# Load Apple stock data
print("Loading AAPL data...")
data = preprocessor.prepare_data(
    ticker="AAPL",
    start_date="2022-01-01",
    features=["returns"],
    normalize="standardize",
)

print(f"\nData prepared:")
print(f"  Train: {data['X_train'].shape}")
print(f"  Test: {data['X_test'].shape}")

In [ ]:
from src.timeseries.ts_model import AdaptiveTimeSeriesLLM
from src.timeseries.ts_trainer import TimeSeriesTrainer

# Create time series model
ts_model = AdaptiveTimeSeriesLLM(
    model_name="gpt2",
    d_input=1,  # Returns only
    d_output=1,
    use_pretrained=True,
    freeze_backbone=False,
    use_temporal_encoding=True,
)

print(f"Time series model created with {sum(p.numel() for p in ts_model.parameters()):,} parameters")

In [ ]:
# Train the time series model
ts_trainer = TimeSeriesTrainer(
    model=ts_model,
    output_dir="./notebook_ts_outputs",
)

print("Training time series model...")
history = ts_trainer.train(
    train_data=data,
    val_data=data,
    epochs=5,  # Small number for demo
    batch_size=32,
    learning_rate=1e-4,
    patience=3,
)

print("\nTraining completed!")

In [ ]:
# Plot training history
plt.figure(figsize=(10, 5))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training History')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Make predictions
predictions = ts_trainer.predict(
    data=data["X_test"],
    batch_size=32,
    steps_ahead=1,
)

# Evaluate
metrics = ts_trainer.evaluate(
    test_data=data,
    batch_size=32,
)

print("\nTest Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value:.6f}")

In [ ]:
# Plot predictions
y_test = data['y_test'][:, 0, 0]
y_pred = predictions[:, 0, 0]

plt.figure(figsize=(12, 6))
plt.plot(y_test[:100], label='Actual', alpha=0.7)
plt.plot(y_pred[:100], label='Predicted', alpha=0.7)
plt.title('AAPL Return Predictions')
plt.xlabel('Time Steps')
plt.ylabel('Normalized Returns')
plt.legend()
plt.grid(True)
plt.show()

## Summary

This notebook demonstrated:
1. Loading a small LLM (GPT-2)
2. Applying LoRA for efficient fine-tuning
3. Preparing datasets
4. Training with the framework
5. Financial time series forecasting

For production use:
- Use larger datasets
- Train for more epochs
- Experiment with different LoRA configurations
- Try different model architectures
- Add more technical indicators for time series